In [1]:
#importing libraries before starting work
import pandas as pd
import numpy as np

#This library is using for the file handling
from pathlib import Path

#for visulaizations
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)


In [2]:
#importing the dataset by using pandas library
# Define the dataset path
DATA_PATH = Path("../data/raw/Dataset.csv")

# Load the dataset
df = pd.read_csv(DATA_PATH)

print("dataset loaded!")

dataset loaded!


In [3]:
#preview of the dataset
df.head()

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


In [4]:
print("Dataset shape(rows,columns):",df.shape)

Dataset shape(rows,columns): (8790, 10)


Inspecting the data quality of the given dataset by performing analyzation

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8790 entries, 0 to 8789
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8790 non-null   str  
 1   type          8790 non-null   str  
 2   title         8790 non-null   str  
 3   director      8790 non-null   str  
 4   country       8790 non-null   str  
 5   date_added    8790 non-null   str  
 6   release_year  8790 non-null   int64
 7   rating        8790 non-null   str  
 8   duration      8790 non-null   str  
 9   listed_in     8790 non-null   str  
dtypes: int64(1), str(9)
memory usage: 686.8 KB


In [6]:
#Identifying the missing values
missing = df.isnull().sum()
print("Columns that have missing values", missing)

Columns that have missing values show_id         0
type            0
title           0
director        0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
dtype: int64


In [7]:
#Duplicated record checking
duplicated = df.duplicated().sum()
print("Duplicated values:", duplicated)

Duplicated values: 0


In [8]:
#need to know about the statistical overview
df.describe(include="all")

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
count,8790,8790,8790,8790,8790,8790,8790.000000,8790,8790,8790
unique,8790,2,8787,4528,86,1713,NaN,14,220,513
top,s1,Movie,9-Feb,Not Given,United States,1/1/2020,NaN,TV-MA,1 Season,"Dramas, International Movies"
freq,1,6126,2,2588,3240,110,NaN,3205,1791,362
mean,NaN,NaN,NaN,NaN,NaN,NaN,2014.183163,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,8.825466,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,1925.000000,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,2013.000000,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,2017.000000,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,2019.000000,NaN,NaN,NaN


## Identifying Hidden Missing Values

Checking whether missing information is represented using text
placeholders instead of standard null values. ( i detected it from the all overview mentioned previously)

In [9]:
#identify possible missing-value placeholders
placeholder_values = [
    "Not Given",
    "Unknown",
    "N/A",
    "None",
    ""
]

for column in df.select_dtypes(include="object").columns:

    count = df[column].astype("string").str.strip().isin(
        placeholder_values
    ).sum()

    if count > 0:
        print(f"{column}: {count} possible missing values")

title: 1 possible missing values
director: 2588 possible missing values
country: 287 possible missing values


C:\Users\User\AppData\Local\Temp\ipykernel_12636\2866813415.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for column in df.select_dtypes(include="object").columns:


In [10]:
# Examine date_added values

print("Sample date values:")
print(df["date_added"].head(10))

print("\nDate column data type:")
print(df["date_added"].dtype)

Sample date values:
0    9/25/2021
1    9/24/2021
2    9/24/2021
3    9/22/2021
4    9/24/2021
5    9/24/2021
6    9/24/2021
7     5/1/2021
8    9/23/2021
9     5/1/2021
Name: date_added, dtype: str

Date column data type:
str


## Detailed Investigation of Missing Values

Identifying the exact placeholder values in columns
that contain incomplete information.

In [11]:
# Investigate missing-value placeholders

for column in ["title", "director", "country"]:

    print(f"\nColumn: {column}")

    print(
        df[column]
        .value_counts()
        .loc[lambda x: x.index.isin(placeholder_values)]
    )


Column: title
title
Unknown    1
Name: count, dtype: int64

Column: director
director
Not Given    2588
Name: count, dtype: int64

Column: country
country
Not Given    287
Name: count, dtype: int64


##Investigating the Unknown value

In [12]:
# Inspect the record with an unknown title

unknown_title = df[
    df["title"].str.strip() == "Unknown"
]

unknown_title

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
1336,s1468,Movie,Unknown,Jaume Collet-Serra,United Kingdom,1/1/2021,2011,PG-13,113 min,"Action & Adventure, Thrillers"


## Handling Hidden Missing Values

Replacing text placeholders with standard missing values
while preserving the original Netflix records.

In [13]:
# Replace missing-value placeholders

df["director"] = df["director"].replace("Not Given", np.nan)

df["country"] = df["country"].replace("Not Given", np.nan)

print("Missing values after cleaning:")
print(df[["director", "country"]].isnull().sum())

Missing values after cleaning:
director    2588
country      287
dtype: int64


In [14]:
print("Sample date values:")
print(df["date_added"].head(10))

print("\nDate column data type:")
print(df["date_added"].dtype)

Sample date values:
0    9/25/2021
1    9/24/2021
2    9/24/2021
3    9/22/2021
4    9/24/2021
5    9/24/2021
6    9/24/2021
7     5/1/2021
8    9/23/2021
9     5/1/2021
Name: date_added, dtype: str

Date column data type:
str


In [15]:
# Replace unknown title with a missing value

df["title"] = df["title"].replace("Unknown", np.nan)

print("Missing values after cleaning:")
print(df[["title", "director", "country"]].isnull().sum())

Missing values after cleaning:
title          1
director    2588
country      287
dtype: int64


In [16]:
# Examine date_added values

print("Sample date values:")
print(df["date_added"].head(10))

print("\nDate column data type:")
print(df["date_added"].dtype)

Sample date values:
0    9/25/2021
1    9/24/2021
2    9/24/2021
3    9/22/2021
4    9/24/2021
5    9/24/2021
6    9/24/2021
7     5/1/2021
8    9/23/2021
9     5/1/2021
Name: date_added, dtype: str

Date column data type:
str


In [17]:
# Convert date_added from string to datetime

df["date_added"] = pd.to_datetime(
    df["date_added"],
    format="%m/%d/%Y",
    errors="coerce"
)

print("Date conversion completed!")

print("\nUpdated data type:")
print(df["date_added"].dtype)

print("\nSample converted dates:")
print(df["date_added"].head(10))

Date conversion completed!

Updated data type:
datetime64[us]

Sample converted dates:
0   2021-09-25
1   2021-09-24
2   2021-09-24
3   2021-09-22
4   2021-09-24
5   2021-09-24
6   2021-09-24
7   2021-05-01
8   2021-09-23
9   2021-05-01
Name: date_added, dtype: datetime64[us]


In [18]:
# Check whether any dates failed to convert

print("Missing dates after conversion:",
      df["date_added"].isnull().sum())

print("\nDate range:")
print("Earliest date:", df["date_added"].min())
print("Latest date:", df["date_added"].max())

Missing dates after conversion: 0

Date range:
Earliest date: 2008-01-01 00:00:00
Latest date: 2021-09-25 00:00:00


In [19]:
# Inspect unique values in categorical columns

categorical_columns = [
    "type",
    "rating",
    "duration"
]

for column in categorical_columns:
    print(f"\nColumn: {column}")
    print(df[column].unique())


Column: type
<StringArray>
['Movie', 'TV Show']
Length: 2, dtype: str

Column: rating
<StringArray>
[   'PG-13',    'TV-MA',    'TV-PG',    'TV-14',    'TV-Y7',     'TV-Y',
       'PG',     'TV-G',        'R',        'G',    'NC-17',       'NR',
 'TV-Y7-FV',       'UR']
Length: 14, dtype: str

Column: duration
<StringArray>
[    '90 min',   '1 Season',     '91 min',    '125 min',  '9 Seasons',
    '104 min',     '87 min',    '127 min',     '76 min',     '71 min',
 ...
    '205 min',    '201 min',    '191 min', '10 Seasons', '17 Seasons',
 '13 Seasons',     '10 min',    '312 min', '12 Seasons', '11 Seasons']
Length: 220, dtype: str


In [20]:
#now we have the idea of the catgories and dividing of the given dataset. before creating or do anything.
# Check missing values in categorical columns

for column in ["type", "rating", "duration"]:

    print(f"\n{column}")
    print("Missing values:", df[column].isnull().sum())

    print("Placeholder values:")
    print(
        df[column]
        .value_counts()
        .loc[lambda x: x.index.isin(
            ["Not Given", "Unknown", "N/A", "NA"]
        )]
    )


type
Missing values: 0
Placeholder values:
Series([], Name: count, dtype: int64)

rating
Missing values: 0
Placeholder values:
Series([], Name: count, dtype: int64)

duration
Missing values: 0
Placeholder values:
Series([], Name: count, dtype: int64)


---
## Step 3 — Handle Missing Values

Strategy:
- `director` and `country` → fill with `"Unknown"` (legitimate to not know).
- `title` → drop the single row (a title-less record is unusable for downstream tasks).

In [21]:
# Fill missing director and country with 'Unknown'
df['director'] = df['director'].fillna('Unknown')
df['country']  = df['country'].fillna('Unknown')

# Drop the single row with no title (unusable for analysis)
rows_before = len(df)
df = df.dropna(subset=['title']).reset_index(drop=True)
rows_after = len(df)

print(f'Rows dropped (null title): {rows_before - rows_after}')
print(f'Remaining rows: {rows_after}')
print('\nNull counts after fill:')
print(df[['title', 'director', 'country']].isnull().sum())

Rows dropped (null title): 1
Remaining rows: 8789

Null counts after fill:
title       0
director    0
country     0
dtype: int64


---
## Step 4 — Transform Columns

### 4a. Parse `duration` → `duration_value` (int) + `duration_unit` (str)

The `duration` column mixes two different units:
- Movies: `"90 min"`, `"125 min"`, …
- TV Shows: `"1 Season"`, `"9 Seasons"`, …

We split this into two clean columns and normalise `"Season"` → `"Seasons"`.

In [22]:
# Split 'duration' into numeric value and unit
duration_split = df['duration'].str.extract(r'(\d+)\s+(.+)')

df['duration_value'] = duration_split[0].astype(int)
df['duration_unit']  = duration_split[1].str.strip()

# Normalise 'Season' → 'Seasons' for consistency
df['duration_unit'] = df['duration_unit'].replace('Season', 'Seasons')

print('duration_unit value counts:')
print(df['duration_unit'].value_counts())
print()
print('Sample parsed durations:')
print(df[['duration', 'duration_value', 'duration_unit']].head(10))

duration_unit value counts:
duration_unit
min        6125
Seasons    2664
Name: count, dtype: int64

Sample parsed durations:
    duration  duration_value duration_unit
0     90 min              90           min
1   1 Season               1       Seasons
2   1 Season               1       Seasons
3     91 min              91           min
4    125 min             125           min
5  9 Seasons               9       Seasons
6    104 min             104           min
7     87 min              87           min
8    127 min             127           min
9     76 min              76           min


### 4b. Extract `year_added` and `month_added` from `date_added`

In [23]:
# Extract year and month from the already-parsed date_added column
df['year_added']  = df['date_added'].dt.year.astype('Int64')   # Int64 handles potential NaT
df['month_added'] = df['date_added'].dt.month.astype('Int64')

print('year_added range:', df['year_added'].min(), '→', df['year_added'].max())
print('month_added range:', df['month_added'].min(), '→', df['month_added'].max())
print()
print(df[['date_added', 'year_added', 'month_added']].head(8))

year_added range: 2008 → 2021
month_added range: 1 → 12

  date_added  year_added  month_added
0 2021-09-25        2021            9
1 2021-09-24        2021            9
2 2021-09-24        2021            9
3 2021-09-22        2021            9
4 2021-09-24        2021            9
5 2021-09-24        2021            9
6 2021-09-24        2021            9
7 2021-05-01        2021            5


### 4c. Whitespace & case standardisation

Strip leading/trailing whitespace from all string columns to prevent silent duplicates.

In [24]:
# Strip whitespace from all string/object columns
str_cols = df.select_dtypes(include='object').columns.tolist()
# Also handle the new pandas str dtype
str_cols += [c for c in df.select_dtypes(include='str').columns if c not in str_cols]

for col in str_cols:
    try:
        df[col] = df[col].str.strip()
    except AttributeError:
        pass

print('Whitespace stripped for columns:', str_cols)

Whitespace stripped for columns: ['show_id', 'type', 'title', 'director', 'country', 'rating', 'duration', 'listed_in', 'duration_unit']


### 4d. Derive `main_country` and `primary_genre`

Both `country` and `listed_in` can contain comma-separated lists.
We extract the **first value** from each as a clean, single-valued feature.

In [25]:
# main_country: first country in the comma-separated list
df['main_country'] = df['country'].str.split(',').str[0].str.strip()

# primary_genre: first genre in the comma-separated listed_in
df['primary_genre'] = df['listed_in'].str.split(',').str[0].str.strip()

print('Top 10 main_country:')
print(df['main_country'].value_counts().head(10))
print()
print('Top 10 primary_genre:')
print(df['primary_genre'].value_counts().head(10))

Top 10 main_country:
main_country
United States     3240
India             1057
United Kingdom     637
Pakistan           421
Unknown            287
Canada             271
Japan              259
South Korea        214
France             213
Spain              182
Name: count, dtype: int64

Top 10 primary_genre:
primary_genre
Dramas                      1599
Comedies                    1210
Action & Adventure           858
Documentaries                829
International TV Shows       773
Children & Family Movies     605
Crime TV Shows               399
Kids' TV                     385
Stand-Up Comedy              334
Horror Movies                275
Name: count, dtype: int64


---
## Step 5 — Final Null Audit

Verify the cleaned dataframe has no unintended nulls.

In [26]:
null_counts = df.isnull().sum()
null_pct    = (null_counts / len(df) * 100).round(2)

audit = pd.DataFrame({'null_count': null_counts, 'null_pct': null_pct})
audit = audit[audit['null_count'] > 0]

if audit.empty:
    print('✅ No null values remain in any column.')
else:
    print('⚠️  Remaining nulls:')
    print(audit)

print(f'\nFinal shape: {df.shape}')
print('Columns:', df.columns.tolist())

✅ No null values remain in any column.

Final shape: (8789, 16)
Columns: ['show_id', 'type', 'title', 'director', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in', 'duration_value', 'duration_unit', 'year_added', 'month_added', 'main_country', 'primary_genre']


---
## Before / After Summary

A concise comparison of the dataset before and after all cleaning steps.

In [27]:
summary = pd.DataFrame({
    'Metric': [
        'Total rows',
        'Total columns',
        'Null values',
        'Duplicate rows',
        'Hidden missing (placeholders)',
        'date_added dtype',
        'duration column',
        'Derived columns added',
    ],
    'Before': [
        8790,
        10,
        0,
        0,
        '2876 ("Not Given" / "Unknown")',
        'str (string)',
        'Mixed string ("90 min", "1 Season")',
        '—',
    ],
    'After': [
        len(df),
        len(df.columns),
        int(df.isnull().sum().sum()),
        int(df.duplicated().sum()),
        '0 (all replaced/filled/dropped)',
        'datetime64[us]',
        'duration_value (int) + duration_unit (str)',
        'year_added, month_added, main_country, primary_genre',
    ]
})

print(summary.to_string(index=False))

                       Metric                              Before                                                After
                   Total rows                                8790                                                 8789
                Total columns                                  10                                                   16
                  Null values                                   0                                                    0
               Duplicate rows                                   0                                                    0
Hidden missing (placeholders)      2876 ("Not Given" / "Unknown")                      0 (all replaced/filled/dropped)
             date_added dtype                        str (string)                                       datetime64[us]
              duration column Mixed string ("90 min", "1 Season")           duration_value (int) + duration_unit (str)
        Derived columns added                   

---
## Step 5b — Export Clean Dataset

Save the cleaned dataframe to `data/processed/netflix_clean.csv`.

In [28]:
output_path = Path('../data/processed/netflix_clean.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(output_path, index=False)

print(f'✅ Clean dataset saved to: {output_path}')
print(f'   Shape : {df.shape}')
print(f'   Size  : {output_path.stat().st_size / 1024:.1f} KB')


✅ Clean dataset saved to: ../data/processed/netflix_clean.csv
   Shape : (8789, 16)
   Size  : 1395.2 KB


---
## Task 1 Summary

**Goal achieved:** Produced a clean, analysis-ready dataset (`netflix_clean.csv`).

### Key decisions made:
1. **Hidden missing values** — `director` and `country` used `"Not Given"` as a placeholder (not standard NaN). These were detected and replaced with proper NaN, then filled with `"Unknown"` to preserve all 8,790 records for analysis.
2. **Title null** — 1 record (show_id `s1468`) had `title = "Unknown"`. This row was **dropped** because a title-less record cannot be used in search, recommendation, or classification tasks.
3. **Date parsing** — `date_added` converted from string (`M/D/YYYY`) to `datetime64`, enabling time-series analysis in Task 2.
4. **Duration split** — The mixed-format `duration` column (`"90 min"` / `"1 Season"`) was split into `duration_value` (integer) and `duration_unit` (categorical), eliminating ambiguity for ML models.
5. **Derived features** — `year_added`, `month_added`, `main_country`, and `primary_genre` added for downstream EDA and modelling.

**Final dataset:** 8,789 rows × 14 columns — no nulls, correct dtypes, fully reproducible.